# HOG màu RGB64, 2.916 chiều + SVM-RBF
- Chạy thử trên ảnh mẫu thuộc train, với một cấu hình và một fold đã lưu.
- Chế độ full cần ảnh đã chuẩn bị. Grid dùng 5 fold trên train và chọn theo accuracy.
- [Kết quả báo cáo](../eval/evaluate.ipynb)

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image, ImageOps

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "papercode").is_dir())
paper = root / "papercode"
prepare = paper / "prepare-dataset"
postures = ["Đi/đứng", "Nằm", "Đang ngã", "Ngồi"]

In [2]:
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import GridSearchCV, ParameterGrid

MODE = "quick"
TRAIN_SOURCE = "mix"  # mcf, upfall hoặc mix
targets = ("mcf", "upfall") if TRAIN_SOURCE == "mix" else (TRAIN_SOURCE,)

def load_rows(target):
    split = pd.read_csv(prepare / f"splits/{target}.csv")
    if MODE == "quick":
        images = prepare / f"dataraw/{target}/examples/images"
        paths = {path.stem: path for path in images.glob("*.png")}
        rows = split[split.sample_id.isin(paths) & split.split.eq("train")].copy()
        rows["path"] = rows.sample_id.map(paths)
        return rows
    folder = prepare / f"prepared/{target}"
    manifest = pd.read_csv(folder / "manifest.csv")
    rows = split.merge(manifest[["sample_id", "image_path"]], on="sample_id", how="left", validate="one_to_one")
    if rows.image_path.isna().any():
        raise ValueError(f"Chuẩn bị đủ ảnh {target} trước khi chạy full")
    rows["path"] = rows.image_path.map(lambda image: folder / image)
    return rows

frames = {target: load_rows(target) for target in targets}
for target, rows in frames.items():
    print(f"{target}: {len(rows)} ảnh · {rows.label.nunique()} lớp")

mcf: 11 ảnh · 4 lớp
upfall: 10 ảnh · 4 lớp


## Trích đặc trưng

In [3]:
from skimage.feature import hog

hog_params = dict(
    orientations=9, pixels_per_cell=(8, 8), cells_per_block=(3, 3),
    block_norm="L2-Hys", channel_axis=-1,
)

def extract_features(paths):
    vectors = []
    for path in paths:
        with Image.open(path) as image:
            rgb128 = ImageOps.exif_transpose(image).convert("RGB").resize((128, 128), Image.Resampling.BILINEAR)
            rgb64 = rgb128.resize((64, 64), Image.Resampling.BILINEAR)
        vectors.append(hog(np.asarray(rgb64, dtype=np.float32), **hog_params))
    return np.asarray(vectors, dtype=np.float32)

## Chọn tham số và huấn luyện

In [4]:
from sklearn.svm import SVC

feature = 'hog'
family = 'svm_rbf'
grid = {'C': [0.1, 1.0, 10.0], 'gamma': ['scale', 'auto']}
estimator = SVC(kernel='rbf', cache_size=256, random_state=1)
display(pd.DataFrame(ParameterGrid(grid)))

,C,gamma
0,0.1,scale
1,0.1,auto
2,1.0,scale
3,1.0,auto
4,10.0,scale
5,10.0,auto


In [5]:
if MODE == "quick":
    paths = pd.concat(frames.values(), ignore_index=True)
    data = extract_features(paths.path.tolist())
    labels = paths.label.to_numpy()
    holdout = paths.cv_fold.eq(0).to_numpy()
    params = next(iter(ParameterGrid(grid)))
    model = estimator.set_params(**params)
    model.fit(data[~holdout], labels[~holdout])
    predictions = model.predict(data[holdout])
    print(f"Chạy thử · tập {TRAIN_SOURCE} · fold 0 · {data.shape[1]:,} đặc trưng")
    print("Tham số:", params, "| fit:", int((~holdout).sum()), "| predict:", int(holdout.sum()))
    print(f"Accuracy mẫu: {accuracy_score(labels[holdout], predictions):.4f}")
else:
    cache = prepare / "prepared/features"
    cache.mkdir(parents=True, exist_ok=True)
    matrices = {}
    for target, rows in frames.items():
        filename = cache / f"{target}_{feature}.npy"
        if not filename.exists():
            np.save(filename, extract_features(rows.path.tolist()))
        matrices[target] = np.load(filename, mmap_mode="r")
    rows = pd.concat(frames.values(), ignore_index=True)
    all_vectors = np.concatenate(list(matrices.values()))
    train = rows.split.eq("train").to_numpy()
    data, labels = all_vectors[train], rows.label.to_numpy()[train]
    folds = rows.cv_fold.to_numpy()[train]
    cv = [(np.flatnonzero(folds != fold), np.flatnonzero(folds == fold)) for fold in range(5)]
    search = GridSearchCV(estimator, grid, cv=cv, scoring="accuracy", n_jobs=1)
    search.fit(data, labels)
    predictions = search.predict(all_vectors[~train])
    print("Tham số tốt nhất:", search.best_params_)
    print("Mean CV accuracy:", search.best_score_)
    print(classification_report(rows.label.to_numpy()[~train], predictions, digits=4))
    import joblib
    output = paper / "train/local-runs"
    output.mkdir(exist_ok=True)
    joblib.dump(search.best_estimator_, output / f"{TRAIN_SOURCE}_{feature}_{family}.joblib", compress=3)

Chạy thử · tập mix · fold 0 · 2,916 đặc trưng
Tham số: {'C': 0.1, 'gamma': 'scale'} | fit: 13 | predict: 8
Accuracy mẫu: 0.2500
